# Gateway API

Ein TCP-Service verteilt Verbindungen; ein HTTP-Gateway kann HTTP-Requests anhand von Pfaden und Gewichten weiterleiten. GatewayClass bezeichnet die Implementierung, der Controller setzt die Konfiguration um, Gateway definiert Listener und HTTPRoute die Routing-Regeln.

![gateway-architecture](gateway-architecture.svg)

## Voraussetzungen

Python-Kernel mit %%bash, kubectl und Zugriff auf den K3s-Trainingscluster. Notebook aus dem Kapitelverzeichnis ausführen. Beispielressourcen müssen frei sein; vorherige Übungen aufräumen. Aktiver Namespace muss default sein.

In [ ]:
%%bash
set -eo pipefail
kubectl config current-context
NS=$(kubectl config view --minify -o jsonpath='{..namespace}')
test "${NS:-default}" = default
kubectl get nodes

## 1. Voraussetzungen

NGINX Gateway Fabric muss gemäß install.md installiert sein. GatewayClass nginx muss Accepted=True zeigen. Port 8080 muss verfügbar sein.

In [ ]:
%%bash
set -eo pipefail
kubectl get gatewayclass nginx -o yaml
kubectl get pods -n nginx-gateway

## 2. Anwendungsversionen

Die Lion- und Ship-Anwendungen werden über getrennte Services bereitgestellt.

In [ ]:
%%bash
set -eo pipefail
cat app-canary/app.yml

In [ ]:
%%bash
set -eo pipefail
kubectl apply -f app-canary/app.yml
kubectl rollout status deployment/current --timeout=180s
kubectl rollout status deployment/previous --timeout=180s
kubectl get svc svc-current svc-previous

## 3. Gateway und Route

Der HTTP-Listener nimmt Requests auf Port 8080 an. PathPrefix / erfasst alle Pfade. Die Gewichte 80 und 20 beziehen sich auf die Backend-Services.

In [ ]:
%%bash
set -eo pipefail
cat app-canary/gtw.yml app-canary/route.yml

In [ ]:
%%bash
set -eo pipefail
kubectl apply -f app-canary/gtw.yml -f app-canary/route.yml
kubectl wait --for=condition=Programmed gateway/gtw-prod-1 --timeout=180s
kubectl get gateway gtw-prod-1
kubectl describe httproute route-canary

## 4. Routing beobachten

Im Route-Status Accepted und ResolvedRefs prüfen. Die ausgegebene Adresse im Browser öffnen, mehrfach mit Shift+Reload laden und Lion/Ship vergleichen. Die Verteilung ist statistisch. Falls die Adresse nicht erreichbar ist, Gateway-Service und dessen externe Bereitstellung prüfen.

In [ ]:
%%bash
set -eo pipefail
IP=$(kubectl get gateway gtw-prod-1 -o jsonpath='{.status.addresses[0].value}')
test -n "$IP"
echo "http://$IP:8080/"
curl -fsS --max-time 10 "http://$IP:8080/"

## 5. Gewichte ändern

Nun beide Backends gleich gewichten. Nach Aktualisierung der Route erneut im Browser testen.

In [ ]:
%%bash
set -eo pipefail
kubectl patch httproute route-canary --type=json -p='[{"op":"replace","path":"/spec/rules/0/backendRefs/0/weight","value":50},{"op":"replace","path":"/spec/rules/0/backendRefs/1/weight","value":50}]'
kubectl describe httproute route-canary

## 6. Aufräumen

Namespace-übergreifendes Routing in app-xnamespace bleibt Vertiefungsmaterial.

In [ ]:
%%bash
set -eo pipefail
kubectl delete -f app-canary/route.yml -f app-canary/gtw.yml -f app-canary/app.yml
kubectl get gateway,httproute

## Fazit

Gateway API ermöglicht HTTP-spezifisches Routing unabhängig von der Anwendung. Gewichte steuern relative Anteile, nicht eine garantierte Reihenfolge.